In [4]:
# --- STEP 1: SETUP & IMPORTS ---
import os
import json
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import KFold, train_test_split
from PIL import Image
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

# Original Drive Paths
JSON_PATH = '/content/drive/MyDrive/project/result.json'
IMAGE_DIR = '/content/drive/MyDrive/project/images/'

# --- STEP 2: EFFICIENTNET SPECIFIC DATASET ---
class BabyFacialLandmarkDataset(Dataset):
    def __init__(self, images_list, img_dir, transform=None):
        self.images_list = images_list
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        item = self.images_list[idx]
        box = item["box"]

        left = float(box["_left"])
        top = float(box["_top"])
        width = float(box["_width"])
        height = float(box["_height"])

        json_file_path = item['_file']
        base_name = os.path.basename(json_file_path)
        img_path = os.path.join(self.img_dir, base_name)

        # Load and crop to facial bounding box
        image = Image.open(img_path).convert('RGB')
        cropped_image = image.crop((left, top, left + width, top + height))

        parts = box["part"]
        if isinstance(parts, dict):
            parts = [parts]
        parts_sorted = sorted(parts, key=lambda x: x["_name"])

        coords = []
        for p in parts_sorted:
            # Map absolute coordinates into relative box space [0.0 - 1.0]
            norm_x = (float(p["_x"]) - left) / width
            norm_y = (float(p["_y"]) - top) / height
            coords.extend([norm_x, norm_y])

        landmarks_tensor = torch.tensor(coords, dtype=torch.float32)

        if self.transform:
            cropped_image = self.transform(cropped_image)

        return cropped_image, landmarks_tensor

# --- STEP 3: TRANSFORMS & DATA PARSING + VERIFICATION ---
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

with open(JSON_PATH, 'r') as f:
    data_json = json.load(f)

all_images_list = data_json['dataset']['images']['image']
if not isinstance(all_images_list, list):
    all_images_list = [all_images_list]

# NEW SAFETY STEP: Scan and remove missing images from the training pool
print("Verifying image files on Google Drive...")
valid_images_list = []
missing_count = 0

for item in all_images_list:
    base_name = os.path.basename(item['_file'])
    img_path = os.path.join(IMAGE_DIR, base_name)

    if os.path.exists(img_path):
        valid_images_list.append(item)
    else:
        missing_count += 1

print(f"Verification Done: Found {len(valid_images_list)} valid images. Skipped {missing_count} missing files.")

# Split dataset using ONLY verified, existing images
cv_list, test_list = train_test_split(valid_images_list, test_size=0.2, random_state=42)

# Pass the verified splits to Datasets
test_dataset = BabyFacialLandmarkDataset(test_list, IMAGE_DIR, transform=transform_pipeline)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# --- STEP 4: EFFICIENTNET-B0 MODEL BUILDER ---
class EfficientNetLandmarkRegressor(nn.Module):
    def __init__(self):
        super(EfficientNetLandmarkRegressor, self).__init__()
        self.network = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        for param in self.network.parameters():
            param.requires_grad = True

        num_ftrs = self.network.classifier[1].in_features

        self.network.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(num_ftrs, 12),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

def build_model():
    model = EfficientNetLandmarkRegressor().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    return model, optimizer

# --- STEP 5: K-FOLD CROSS VALIDATION LOOP ---
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

results = {"Fold": [], "Val_MSE": [], "Val_MAE": [], "Test_MSE": [], "Test_MAE": []}
fold_no = 1

def evaluate_model(model, dataloader):
    model.eval()
    mse_loss = nn.MSELoss()
    mae_loss = nn.L1Loss()

    total_mse, total_mae, count = 0.0, 0.0, 0
    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)

            total_mse += mse_loss(outputs, targets).item() * images.size(0)
            total_mae += mae_loss(outputs, targets).item() * images.size(0)
            count += images.size(0)

    return total_mse / count, total_mae / count

cv_list_np = np.array(cv_list)

for train_idx, val_idx in kf.split(cv_list_np):
    print(f"\n{'='*20} TRAINING FOLD {fold_no} / {n_splits} {'='*20}")

    train_fold_data = cv_list_np[train_idx].tolist()
    val_fold_data = cv_list_np[val_idx].tolist()

    train_dataset = BabyFacialLandmarkDataset(train_fold_data, IMAGE_DIR, transform=transform_pipeline)
    val_dataset = BabyFacialLandmarkDataset(val_fold_data, IMAGE_DIR, transform=transform_pipeline)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    torch.cuda.empty_cache()
    gc.collect()
    model, optimizer = build_model()
    criterion = nn.MSELoss()

    epochs = 15
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        print(f"Epoch {epoch+1}/{epochs} | Train MSE: {running_loss/len(train_dataset):.4f}")

    val_mse, val_mae = evaluate_model(model, val_loader)
    test_mse, test_mae = evaluate_model(model, test_loader)

    results["Fold"].append(f"Fold {fold_no}")
    results["Val_MSE"].append(val_mse)
    results["Val_MAE"].append(val_mae)
    results["Test_MSE"].append(test_mse)
    results["Test_MAE"].append(test_mae)

    del model, optimizer, train_loader, val_loader
    fold_no += 1

# --- STEP 6: PRINT SCORES TABLE ---
print("\n\n" + "#"*70 + "\n FINAL CROSS-VALIDATION RESULTS (EFFICIENTNET-B0) \n" + "#"*70)

results["Fold"].append("Mean ± Std")
for metric in ["Val_MSE", "Val_MAE", "Test_MSE", "Test_MAE"]:
    mean_val, std_val = np.mean(results[metric]), np.std(results[metric])
    results[metric].append(f"{mean_val:.4f} ± {std_val:.4f}")

df_results = pd.DataFrame(results).set_index("Fold")
print("\n")
print(df_results.to_markdown())
print("\n" + "#"*70)

Mounted at /content/drive
Using Device: cuda
Verifying image files on Google Drive...
Verification Done: Found 243 valid images. Skipped 1 missing files.

==================== TRAINING FOLD 1 / 5 ====================
Epoch 1/15 | Train MSE: 0.0384
Epoch 2/15 | Train MSE: 0.0273
Epoch 3/15 | Train MSE: 0.0198
Epoch 4/15 | Train MSE: 0.0157
Epoch 5/15 | Train MSE: 0.0131
Epoch 6/15 | Train MSE: 0.0115
Epoch 7/15 | Train MSE: 0.0106
Epoch 8/15 | Train MSE: 0.0093
Epoch 9/15 | Train MSE: 0.0088
Epoch 10/15 | Train MSE: 0.0069
Epoch 11/15 | Train MSE: 0.0068
Epoch 12/15 | Train MSE: 0.0057
Epoch 13/15 | Train MSE: 0.0053
Epoch 14/15 | Train MSE: 0.0042
Epoch 15/15 | Train MSE: 0.0044

==================== TRAINING FOLD 2 / 5 ====================
Epoch 1/15 | Train MSE: 0.0367
Epoch 2/15 | Train MSE: 0.0257
Epoch 3/15 | Train MSE: 0.0187
Epoch 4/15 | Train MSE: 0.0147
Epoch 5/15 | Train MSE: 0.0120
Epoch 6/15 | Train MSE: 0.0109
Epoch 7/15 | Train MSE: 0.0102
Epoch 8/15 | Train MSE: 0.0080
E

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
# --- STEP 1: SETUP & IMPORTS ---
import os
import json
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import KFold, train_test_split
from PIL import Image
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

# Paths
JSON_PATH = '/content/drive/MyDrive/project/result.json'
DRIVE_IMAGE_DIR = '/content/drive/MyDrive/project/images/'
LOCAL_IMAGE_DIR = '/content/local_images/'

# --- STEP 2: PREVENT DISCONNECTS (COPY DATA LOCALLY) ---
if not os.path.exists(LOCAL_IMAGE_DIR):
    print("Copying images from Google Drive to local storage for faster, stable training...")
    shutil.copytree(DRIVE_IMAGE_DIR, LOCAL_IMAGE_DIR)
    print("Copy complete!")
else:
    print("Images are already stored locally.")

# --- STEP 3: DATASET DEFINITION ---
class BabyFacialLandmarkDataset(Dataset):
    def __init__(self, images_list, img_dir, transform=None):
        self.images_list = images_list
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        item = self.images_list[idx]
        box = item["box"]

        left = float(box["_left"])
        top = float(box["_top"])
        width = float(box["_width"])
        height = float(box["_height"])

        json_file_path = item['_file']
        base_name = os.path.basename(json_file_path)
        img_path = os.path.join(self.img_dir, base_name)

        # Load and crop to facial bounding box
        image = Image.open(img_path).convert('RGB')
        cropped_image = image.crop((left, top, left + width, top + height))

        parts = box["part"]
        if isinstance(parts, dict):
            parts = [parts]
        parts_sorted = sorted(parts, key=lambda x: x["_name"])

        coords = []
        for p in parts_sorted:
            # Map absolute coordinates into relative box space [0.0 - 1.0]
            norm_x = (float(p["_x"]) - left) / width
            norm_y = (float(p["_y"]) - top) / height
            coords.extend([norm_x, norm_y])

        landmarks_tensor = torch.tensor(coords, dtype=torch.float32)

        if self.transform:
            cropped_image = self.transform(cropped_image)

        return cropped_image, landmarks_tensor

# --- STEP 4: VERIFICATION & DATA SPLITTING ---
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

with open(JSON_PATH, 'r') as f:
    data_json = json.load(f)

all_images_list = data_json['dataset']['images']['image']
if not isinstance(all_images_list, list):
    all_images_list = [all_images_list]

print("Verifying image files...")
valid_images_list = []
missing_count = 0

for item in all_images_list:
    base_name = os.path.basename(item['_file'])
    img_path = os.path.join(LOCAL_IMAGE_DIR, base_name)

    if os.path.exists(img_path):
        valid_images_list.append(item)
    else:
        missing_count += 1

print(f"Verification Done: Found {len(valid_images_list)} valid images. Skipped {missing_count} files.")

cv_list, test_list = train_test_split(valid_images_list, test_size=0.2, random_state=42)

test_dataset = BabyFacialLandmarkDataset(test_list, LOCAL_IMAGE_DIR, transform=transform_pipeline)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# --- STEP 5: RESNET-V1 MODEL BUILDER ---
class ResNetV1LandmarkRegressor(nn.Module):
    def __init__(self):
        super(ResNetV1LandmarkRegressor, self).__init__()
        # Load standard ResNet-50 (v1) backbone
        self.network = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        for param in self.network.parameters():
            param.requires_grad = True # Keeping it true for fine-tuning

        num_ftrs = self.network.fc.in_features

        # Replace the classifier head for 12 coordinate outputs
        self.network.fc = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(num_ftrs, 12),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

def build_model():
    model = ResNetV1LandmarkRegressor().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    return model, optimizer

# --- STEP 6: K-FOLD CROSS VALIDATION LOOP ---
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

results = {"Fold": [], "Val_MSE": [], "Val_MAE": [], "Test_MSE": [], "Test_MAE": []}
fold_no = 1

def evaluate_model(model, dataloader):
    model.eval()
    mse_loss = nn.MSELoss()
    mae_loss = nn.L1Loss()

    total_mse, total_mae, count = 0.0, 0.0, 0
    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)

            total_mse += mse_loss(outputs, targets).item() * images.size(0)
            total_mae += mae_loss(outputs, targets).item() * images.size(0)
            count += images.size(0)

    return total_mse / count, total_mae / count

cv_list_np = np.array(cv_list)

for train_idx, val_idx in kf.split(cv_list_np):
    print(f"\n{'='*20} TRAINING FOLD {fold_no} / {n_splits} {'='*20}")

    train_fold_data = cv_list_np[train_idx].tolist()
    val_fold_data = cv_list_np[val_idx].tolist()

    train_dataset = BabyFacialLandmarkDataset(train_fold_data, LOCAL_IMAGE_DIR, transform=transform_pipeline)
    val_dataset = BabyFacialLandmarkDataset(val_fold_data, LOCAL_IMAGE_DIR, transform=transform_pipeline)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    torch.cuda.empty_cache()
    gc.collect()
    model, optimizer = build_model()
    criterion = nn.MSELoss()

    epochs = 15
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        print(f"Epoch {epoch+1}/{epochs} | Train MSE: {running_loss/len(train_dataset):.4f}")

    val_mse, val_mae = evaluate_model(model, val_loader)
    test_mse, test_mae = evaluate_model(model, test_loader)

    results["Fold"].append(f"Fold {fold_no}")
    results["Val_MSE"].append(val_mse)
    results["Val_MAE"].append(val_mae)
    results["Test_MSE"].append(test_mse)
    results["Test_MAE"].append(test_mae)

    del model, optimizer, train_loader, val_loader
    fold_no += 1

# --- STEP 7: PRINT SCORES TABLE ---
print("\n\n" + "#"*70 + "\n FINAL CROSS-VALIDATION RESULTS (RESNET V1) \n" + "#"*70)

results["Fold"].append("Mean ± Std")
for metric in ["Val_MSE", "Val_MAE", "Test_MSE", "Test_MAE"]:
    mean_val, std_val = np.mean(results[metric]), np.std(results[metric])
    results[metric].append(f"{mean_val:.4f} ± {std_val:.4f}")

df_results = pd.DataFrame(results).set_index("Fold")
print("\n")
print(df_results.to_markdown())
print("\n" + "#"*70)

Mounted at /content/drive
Using Device: cuda
Images are already stored locally.
Verifying image files...
Verification Done: Found 243 valid images. Skipped 1 files.

==================== TRAINING FOLD 1 / 5 ====================
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]


Epoch 1/15 | Train MSE: 0.0331
Epoch 2/15 | Train MSE: 0.0135
Epoch 3/15 | Train MSE: 0.0078
Epoch 4/15 | Train MSE: 0.0048
Epoch 5/15 | Train MSE: 0.0045
Epoch 6/15 | Train MSE: 0.0043
Epoch 7/15 | Train MSE: 0.0041
Epoch 8/15 | Train MSE: 0.0035
Epoch 9/15 | Train MSE: 0.0035
Epoch 10/15 | Train MSE: 0.0034
Epoch 11/15 | Train MSE: 0.0034
Epoch 12/15 | Train MSE: 0.0032
Epoch 13/15 | Train MSE: 0.0026
Epoch 14/15 | Train MSE: 0.0034
Epoch 15/15 | Train MSE: 0.0027

==================== TRAINING FOLD 2 / 5 ====================
Epoch 1/15 | Train MSE: 0.0275
Epoch 2/15 | Train MSE: 0.0118
Epoch 3/15 | Train MSE: 0.0067
Epoch 4/15 | Train MSE: 0.0059
Epoch 5/15 | Train MSE: 0.0041
Epoch 6/15 | Train MSE: 0.0040
Epoch 7/15 | Train MSE: 0.0036
Epoch 8/15 | Train MSE: 0.0035
Epoch 9/15 | Train MSE: 0.0041
Epoch 10/15 | Train MSE: 0.0033
Epoch 11/15 | Train MSE: 0.0030
Epoch 12/15 | Train MSE: 0.0030
Epoch 13/15 | Train MSE: 0.0033
Epoch 14/15 | Train MSE: 0.0026
Epoch 15/15 | Train MSE: 0.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import json
import cv2
import gc
import numpy as np
import tensorflow as tf
import keras
import pandas as pd
import albumentations as A
from sklearn.model_selection import KFold, train_test_split
from google.colab import drive

# --- STEP 1: MOUNT DRIVE & SET PATHS ---
drive.mount('/content/drive')

JSON_FILE = '/content/drive/MyDrive/project/result.json'
IMAGE_DIR = '/content/drive/MyDrive/project/images/'
IMG_SIZE = 224

# --- STEP 2: DEFINE YOUR AUGMENTOR ---
augmentor = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.06, rotate_limit=12, p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3)
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

# --- STEP 3: PARSE RAW DATA ---
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

X_raw, y_raw = [], []
items = data['dataset']['images']['image']
if not isinstance(items, list):
    items = [items]

print("Parsing JSON dataset hierarchy...")
for item in items:
    img_name = item['_file']
    img_path = os.path.join(IMAGE_DIR, img_name)

    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_w, orig_h = float(item['_width']), float(item['_height'])

        parts_list = item['box']['part']
        if not isinstance(parts_list, list):
            parts_list = [parts_list]

        sorted_parts = sorted(parts_list, key=lambda k: k['_name'])

        pts_absolute = []
        for part in sorted_parts:
            pts_absolute.append((float(part['_x']), float(part['_y'])))

        X_raw.append((img, orig_w, orig_h))
        y_raw.append(pts_absolute)

indices = list(range(len(X_raw)))
cv_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42)

# --- STEP 4: DEFINE MODEL BUILDER (VGGFace2 ResNet-50) ---
def build_vgg_resnet_model():
    full_vgg_model = keras.saving.load_model("hf://logasja/VGGFace2")
    base_model = keras.Model(inputs=full_vgg_model.input, outputs=full_vgg_model.layers[-2].output)
    base_model.trainable = True

    preprocess_model = keras.Sequential([
        keras.layers.Rescaling(1./255),
        keras.layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])
    ])

    model = keras.Sequential([
        keras.Input(shape=(224, 224, 3)),
        preprocess_model,
        base_model,
        keras.layers.Dense(256, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(12, activation='sigmoid')
    ])

    # CHANGED: Compile with 'mse' to get Mean Squared Error
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-5),
        loss='mse',
        metrics=['mae']
    )
    return model

# --- STEP 5: PREPARE FINAL UNSEEN TEST DATA ---
X_test_final, y_test_final = [], []
for idx in test_indices:
    img, w, h = X_raw[idx]
    X_test_final.append(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
    norm_pts = []
    for (x, y) in y_raw[idx]:
        norm_pts.extend([x / w, y / h])
    y_test_final.append(norm_pts)
X_test_final = np.array(X_test_final, dtype='float32')
y_test_final = np.array(y_test_final, dtype='float32')

# --- STEP 6: K-FOLD SYSTEM WITH INNER AUGMENTATION ---
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# CHANGED: Dictionary keys now track MSE
results = {"Fold": [], "Val_MSE": [], "Val_MAE": [], "Test_MSE": [], "Test_MAE": []}
fold_no = 1

cv_indices = np.array(cv_indices)

for train_fold_idx, val_fold_idx in kf.split(cv_indices):
    print(f"\n==================== TRAINING FOLD {fold_no} / {n_splits} ====================")

    actual_train_idxs = cv_indices[train_fold_idx]
    actual_val_idxs = cv_indices[val_fold_idx]

    X_val, y_val = [], []
    for idx in actual_val_idxs:
        img, w, h = X_raw[idx]
        X_val.append(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
        norm_pts = []
        for (x, y) in y_raw[idx]:
            norm_pts.extend([x / w, y / h])
        y_val.append(norm_pts)
    X_val = np.array(X_val, dtype='float32')
    y_val = np.array(y_val, dtype='float32')

    X_train, y_train = [], []
    for idx in actual_train_idxs:
        img, w, h = X_raw[idx]
        pts = y_raw[idx]

        X_train.append(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
        base_pts = []
        for (x, y) in pts:
            base_pts.extend([x / w, y / h])
        y_train.append(base_pts)

        for _ in range(3):
            try:
                augmented = augmentor(image=img, keypoints=pts)
                aug_img = cv2.resize(augmented['image'], (IMG_SIZE, IMG_SIZE))
                aug_pts = []
                for kp in augmented['keypoints']:
                    aug_pts.extend([kp[0] / w, kp[1] / h])
                if len(aug_pts) == 12:
                    X_train.append(aug_img)
                    y_train.append(aug_pts)
            except Exception:
                continue
    X_train = np.array(X_train, dtype='float32')
    y_train = np.array(y_train, dtype='float32')

    keras.backend.clear_session()
    gc.collect()

    fold_model = build_vgg_resnet_model()

    fold_model.fit(
        X_train, y_train,
        epochs=15,
        batch_size=8,
        validation_data=(X_val, y_val),
        verbose=1
    )

    # Because we compiled with loss='mse', val_loss is now MSE!
    val_loss, val_mae = fold_model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_mae = fold_model.evaluate(X_test_final, y_test_final, verbose=0)

    results["Fold"].append(f"Fold {fold_no}")
    results["Val_MSE"].append(val_loss)
    results["Val_MAE"].append(val_mae)
    results["Test_MSE"].append(test_loss)
    results["Test_MAE"].append(test_mae)

    del fold_model
    fold_no += 1

# --- STEP 7: PRINT SCORES TABLE ---
print("\n\n" + "#"*70 + "\n FINAL CROSS-VALIDATION RESULTS \n" + "#"*70)

results["Fold"].append("Mean ± Std")

# CHANGED: Loop through the new MSE keys
for metric in ["Val_MSE", "Val_MAE", "Test_MSE", "Test_MAE"]:
    mean_val, std_val = np.mean(results[metric]), np.std(results[metric])
    results[metric].append(f"{mean_val:.4f} ± {std_val:.4f}")

df_results = pd.DataFrame(results).set_index("Fold")
print("\n")
print(df_results.to_markdown())
print("\n" + "#"*70)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Parsing JSON dataset hierarchy...

==================== TRAINING FOLD 1 / 5 ====================


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 1/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 28s 397ms/step - loss: 0.0156 - mae: 0.0997 - val_loss: 0.0298 - val_mae: 0.1489
Epoch 2/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 284ms/step - loss: 0.0091 - mae: 0.0754 - val_loss: 0.0289 - val_mae: 0.1470
Epoch 3/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 289ms/step - loss: 0.0069 - mae: 0.0660 - val_loss: 0.0286 - val_mae: 0.1456
Epoch 4/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 285ms/step - loss: 0.0062 - mae: 0.0613 - val_loss: 0.0281 - val_mae: 0.1440
Epoch 5/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 281ms/step - loss: 0.0057 - mae: 0.0599 - val_loss: 0.0259 - val_mae: 0.1381
Epoch 6/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 279ms/step - loss: 0.0052 - mae: 0.0563 - val_loss: 0.0238 - val_mae: 0.1324
Epoch 7/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 277ms/step - loss: 0.0046 - mae: 0.0535 - val_loss: 0.0206 - val_mae: 0.1241
Epoch 8/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 279ms/step - loss: 0.0039 - mae: 0.0492 - val_loss: 0.0211 - val_mae: 0.1243
Epoch 9/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 26s 329ms/step - loss: 0.0220 - mae: 0.1221 - val_loss: 0.0193 - val_mae: 0.1177
Epoch 2/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 282ms/step - loss: 0.0109 - mae: 0.0842 - val_loss: 0.0180 - val_mae: 0.1136
Epoch 3/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 288ms/step - loss: 0.0072 - mae: 0.0677 - val_loss: 0.0177 - val_mae: 0.1125
Epoch 4/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 290ms/step - loss: 0.0071 - mae: 0.0676 - val_loss: 0.0168 - val_mae: 0.1093
Epoch 5/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 284ms/step - loss: 0.0069 - mae: 0.0659 - val_loss: 0.0165 - val_mae: 0.1086
Epoch 6/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/step - loss: 0.0057 - mae: 0.0603 - val_loss: 0.0151 - val_mae: 0.1039
Epoch 7/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 278ms/step - loss: 0.0053 - mae: 0.0554 - val_loss: 0.0143 - val_mae: 0.1006
Epoch 8/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 278ms/step - loss: 0.0049 - mae: 0.0552 - val_loss: 0.0131 - val_mae: 0.0962
Epoch 9/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 25s 316ms/step - loss: 0.0257 - mae: 0.1325 - val_loss: 0.0163 - val_mae: 0.1078
Epoch 2/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 283ms/step - loss: 0.0112 - mae: 0.0844 - val_loss: 0.0155 - val_mae: 0.1051
Epoch 3/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 289ms/step - loss: 0.0074 - mae: 0.0685 - val_loss: 0.0150 - val_mae: 0.1031
Epoch 4/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 289ms/step - loss: 0.0068 - mae: 0.0645 - val_loss: 0.0141 - val_mae: 0.0997
Epoch 5/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/step - loss: 0.0058 - mae: 0.0597 - val_loss: 0.0136 - val_mae: 0.0976
Epoch 6/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 278ms/step - loss: 0.0055 - mae: 0.0576 - val_loss: 0.0125 - val_mae: 0.0939
Epoch 7/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 277ms/step - loss: 0.0056 - mae: 0.0586 - val_loss: 0.0115 - val_mae: 0.0894
Epoch 8/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 278ms/step - loss: 0.0044 - mae: 0.0521 - val_loss: 0.0105 - val_mae: 0.0862
Epoch 9/15
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/